# ASTRA — Module 1 (OGM) + Module 2 (DLC) Debug Notebook

Notebook này dùng để:
- Debug Module 1 (OGM): bbox detection bằng GroundingDINO
- Debug Module 2 (DLC): depth estimation bằng Depth-Anything-V2
- Visualize kết quả intermediate trên các sample từ `test_objects_last.json`

## Cell 1 — Setup ASTRA repo

In [ ]:
import os

ASTRA_DIR = '/kaggle/working/ASTRA'

if not os.path.exists(ASTRA_DIR):
    !git clone https://github.com/tuikhongtenbo/ASTRA.git
else:
    !cd /kaggle/working/ASTRA && git pull

%cd {ASTRA_DIR}
print('CWD:', os.getcwd())

## Cell 2 — Clone + Install GroundingDINO

In [ ]:
import os
import sys

GD_DIR = '/kaggle/working/ASTRA/GroundingDINO'

if not os.path.exists(GD_DIR):
    print('Cloning GroundingDINO...')
    !git clone https://github.com/IDEA-Research/GroundingDINO.git {GD_DIR}
else:
    print('GroundingDINO already cloned.')

# Inject vào sys.path ngay lập tức
if GD_DIR not in sys.path:
    sys.path.insert(0, GD_DIR)
    print(f'Injected {GD_DIR} to sys.path')

# Install GroundingDINO bằng cách đúng
print('Installing GroundingDINO package...')
!pip install -q -e {GD_DIR}

print('Done!')

## Cell 3 — Tải weights GroundingDINO

In [ ]:
import os

WEIGHTS_DIR = '/kaggle/working/ASTRA/GroundingDINO/weights'
CKPT_PATH = os.path.join(WEIGHTS_DIR, 'groundingdino_swint_ogc.pth')
CFG_PATH = '/kaggle/working/ASTRA/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(CKPT_PATH):
    print('Downloading weights...')
    !wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth \
        -O {CKPT_PATH}
    print('Done!')
else:
    print(f'Weights already exist: {CKPT_PATH}')

print(f'Config: {CFG_PATH}')
print(f'Exists: {os.path.exists(CFG_PATH)}')

## Cell 4 — Install requirements

In [ ]:
!pip install -q -r /kaggle/working/ASTRA/requirements_base.txt

## Cell 5 — Patch GroundingDINO cho transformers >= 4.31

In [ ]:
"""
transformers >= 4.31 thay đổi cách kế thừa BertPreTrainedModel.
GroundingDINO gọi self.get_head_mask() nhưng method này bị 'mất' 
trong một số class con. Patch thủ công trước khi load.
"""
import sys
sys.path.insert(0, '/kaggle/working/ASTRA/GroundingDINO')

try:
    import transformers.models.bert.modeling_bert as bert_module
    from transformers.modeling_utils import PreTrainedModel
    if not hasattr(bert_module.BertPreTrainedModel, 'get_head_mask'):
        bert_module.BertPreTrainedModel.get_head_mask = PreTrainedModel.get_head_mask
        print('[Patch] BertPreTrainedModel.get_head_mask injected')
    if not hasattr(bert_module.BertModel, 'get_head_mask'):
        bert_module.BertModel.get_head_mask = PreTrainedModel.get_head_mask
        print('[Patch] BertModel.get_head_mask injected')
    else:
        print('[Patch] get_head_mask already exists — no patch needed')
except Exception as e:
    print(f'[Patch] Warning: {e}')

## Cell 6 — Load GroundingDINO model

In [ ]:
import torch
from groundingdino.util.inference import load_model, load_image, predict, annotate

# Patch lại sau khi import (BertModelWarper được tạo sau khi import)
try:
    import groundingdino.models.GroundingDINO.bertwarper as bw
    from transformers.modeling_utils import PreTrainedModel
    if not hasattr(bw.BertModelWarper, 'get_head_mask'):
        bw.BertModelWarper.get_head_mask = PreTrainedModel.get_head_mask
        print('[Patch] BertModelWarper.get_head_mask injected')
except Exception as e:
    print(f'[Patch bertwarper] {e}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CFG_PATH = '/kaggle/working/ASTRA/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'
CKPT_PATH = '/kaggle/working/ASTRA/GroundingDINO/weights/groundingdino_swint_ogc.pth'

print(f'Loading GroundingDINO on {DEVICE}...')
gd_model = load_model(CFG_PATH, CKPT_PATH, device=DEVICE)
gd_model.eval()
print('GroundingDINO loaded!')

## Cell 7 — Load Depth-Anything-V2 model

In [ ]:
from depth_anything_v2.dpt import DepthAnythingV2
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

depth_model = DepthAnythingV2(
    encoder='vitl',   # Small model
    features=256,
    out_channels=[256, 512, 1024, 1024]
)
depth_model = depth_model.to(DEVICE).eval()
print(f'Depth-Anything-V2-Small loaded on {DEVICE}')

## Cell 8 — Chọn samples để debug

In [ ]:
import json, random

EXTRACTION_FILE = '/kaggle/working/ASTRA/data/test_objects_last.json'

with open(EXTRACTION_FILE, 'r', encoding='utf-8') as f:
    all_records = json.load(f)

# Lọc đa dạng: viewer True/False, conf cao/thấp, halluc
groups = {'high_conf_viewer': [], 'high_conf_nonviewer': [], 'low_conf': [], 'hallucination': []}
for r in all_records:
    is_viewer = bool(r.get('O2_is_viewer', False))
    conf = r.get('confidence', 0.0)
    halluc = r.get('O1_hallucinated', False) or r.get('O2_hallucinated', False)
    if halluc:
        groups['hallucination'].append(r)
    elif conf < 0.6:
        groups['low_conf'].append(r)
    elif is_viewer:
        groups['high_conf_viewer'].append(r)
    else:
        groups['high_conf_nonviewer'].append(r)

NUM_SAMPLES = 9
selected = []
per_group = max(2, NUM_SAMPLES // len(groups))
for name, recs in groups.items():
    selected.extend(recs[:per_group])
random.shuffle(selected)
selected = selected[:NUM_SAMPLES]

print(f'Đã chọn {len(selected)} samples:')
for r in selected:
    print(f'  id={r["id"]}: viewer={r.get("O2_is_viewer")}, conf={r.get("confidence",0):.2f}, '
          f'halluc={r.get("O1_hallucinated",False) or r.get("O2_hallucinated",False)}')

## Cell 9 — Hàm tìm đường dẫn ảnh

In [ ]:
import os

IMAGE_DIR_CANDIDATES = [
    '/kaggle/working/ASTRA/data/images/relevant_images',
    '/kaggle/working/ASTRA/data/images',
    '/kaggle/working/relevant_images',
]

def find_image_path(image_name: str) -> str | None:
    for base in IMAGE_DIR_CANDIDATES:
        for name in [image_name, image_name.replace('.jpg', '.png')]:
            p = os.path.join(base, name)
            if os.path.exists(p):
                return p
    return None

# Test thử
test_img = selected[0].get('image', '') if selected else ''
p = find_image_path(test_img)
print(f'Test: {test_img} -> {p}')

## Cell 10 — Module 1 (OGM): Detect bbox bằng GroundingDINO

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from groundingdino.util.inference import load_image, predict

BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def detect_entity(img_path: str, text: str, box_thr=BOX_THRESHOLD, text_thr=TEXT_THRESHOLD):
    """
    Detect 1 entity trên ảnh bằng GroundingDINO.
    Trả về (bbox_norm [x1,y1,x2,y2], score) hoặc (None, 0.0)
    """
    if not text.strip():
        return None, 0.0
    try:
        image_source, image_tensor = load_image(img_path)
        caption = text.strip().rstrip('.') + '.'
        boxes, logits, phrases = predict(
            model=gd_model,
            image=image_tensor,
            caption=caption,
            box_threshold=box_thr,
            text_threshold=text_thr,
            device=DEVICE,
        )
        if boxes is None or len(boxes) == 0:
            return None, 0.0
        # Lấy box có score cao nhất
        best_idx = int(torch.argmax(logits))
        score = float(logits[best_idx])
        # boxes là [cx, cy, w, h] normalized
        cx, cy, bw, bh = boxes[best_idx].tolist()
        x1 = max(0.0, cx - bw / 2)
        y1 = max(0.0, cy - bh / 2)
        x2 = min(1.0, cx + bw / 2)
        y2 = min(1.0, cy + bh / 2)
        return [x1, y1, x2, y2], score
    except Exception as e:
        print(f'  [detect_entity] Error: {e}')
        return None, 0.0


def draw_bbox_marks(img: Image.Image, bbox1, bbox2, name1='O1', name2='O2') -> Image.Image:
    """Vẽ Set-of-Mark [1] và [2] lên ảnh."""
    img = img.copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    font_size = max(14, min(28, int(min(w, h) * 0.04)))
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', font_size)
    except:
        font = ImageFont.load_default()

    colors = [(255, 60, 60), (30, 90, 255)]
    labels = [f'[1] {name1}', f'[2] {name2}']
    bboxes = [bbox1, bbox2]

    for bbox, color, label in zip(bboxes, colors, labels):
        if bbox is None:
            continue
        x1, y1, x2, y2 = bbox
        px1, py1 = int(x1 * w), int(y1 * h)
        px2, py2 = int(x2 * w), int(y2 * h)
        draw.rectangle([px1, py1, px2, py2], outline=color, width=3)
        draw.text((px1 + 4, py1 + 4), label, fill=color, font=font)

    return img


print('Module 1 functions defined.')

## Cell 11 — Module 2 (DLC): Depth estimation

In [ ]:
import numpy as np
import torch
from PIL import Image
from torchvision import transforms

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def compute_depth_map(pil_image: Image.Image, target_size: int = 518) -> np.ndarray:
    """Tính depth map từ ảnh PIL. Trả về numpy array [H, W], nhỏ = gần."""
    transform = transforms.Compose([
        transforms.Resize((target_size, target_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    img_tensor = transform(pil_image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        depth = depth_model(img_tensor).squeeze().cpu().numpy()
    # Resize về kích thước gốc
    w, h = pil_image.size
    depth_img = Image.fromarray((depth * 255 / depth.max()).astype(np.uint8))
    depth_img = depth_img.resize((w, h), Image.BILINEAR)
    depth_map = np.array(depth_img).astype(np.float32) / 255.0
    if depth_map.max() > depth_map.min():
        depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
    return 1.0 - depth_map  # nhỏ = gần camera


def get_mean_depth_in_bbox(depth_map: np.ndarray, bbox, image_size: tuple) -> float:
    """Tính mean depth trong bbox. bbox=[x1,y1,x2,y2] normalized."""
    if bbox is None:
        return float(np.median(depth_map))
    h, w = image_size
    x1, y1, x2, y2 = bbox
    px1, py1 = max(0, int(x1 * w)), max(0, int(y1 * h))
    px2, py2 = min(w, int(x2 * w)), min(h, int(y2 * h))
    if px2 <= px1 or py2 <= py1:
        return float(np.median(depth_map))
    region = depth_map[py1:py2, px1:px2]
    return float(np.mean(region)) if region.size > 0 else float(np.median(depth_map))


def depth_map_to_heatmap(depth_map: np.ndarray) -> Image.Image:
    """Chuyển depth map sang ảnh heatmap màu."""
    import matplotlib.pyplot as plt
    import io
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(depth_map, cmap='viridis')
    plt.colorbar(im, ax=ax, label='Depth (nhỏ=gần)')
    ax.set_title('Depth Map')
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    plt.close()
    buf.seek(0)
    return Image.open(buf)


print('Module 2 functions defined.')

## Cell 12 — Chạy pipeline trên tất cả samples

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

OUTPUT_DIR = '/kaggle/working/ASTRA/output/debug_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = []

for idx, record in enumerate(selected):
    sid = str(record.get('id', idx))
    img_name = record.get('image', '')
    o1_name = record.get('Object', {}).get('O1', '')
    o2_name = record.get('Object', {}).get('O2', '')
    is_viewer = bool(record.get('O2_is_viewer', False))
    conf = record.get('confidence', 0.0)
    halluc = record.get('O1_hallucinated', False) or record.get('O2_hallucinated', False)
    gating_pass = (conf >= 0.6) and not halluc

    print(f'\n[{idx+1}/{len(selected)}] id={sid} | img={img_name}')
    print(f'  O1: {o1_name} | O2: {o2_name} | viewer={is_viewer} | conf={conf:.2f}')

    # Tìm ảnh
    img_path = find_image_path(img_name)
    if img_path is None:
        print(f'  [ERROR] Ảnh không tìm thấy: {img_name}')
        results.append({'id': sid, 'error': 'image_not_found'})
        continue

    img = Image.open(img_path).convert('RGB')
    img_w, img_h = img.size

    # ── Module 1: Bbox Detection ──
    marks_ok = False
    box_o1, score_o1 = None, 0.0
    box_o2, score_o2 = None, 0.0
    marked_img = img.copy()

    if gating_pass:
        print(f'  [M1] Detecting O1: "{o1_name}"')
        box_o1, score_o1 = detect_entity(img_path, o1_name)
        print(f'       → score={score_o1:.3f}, box={box_o1}')

        if not is_viewer and o2_name:
            print(f'  [M1] Detecting O2: "{o2_name}"')
            box_o2, score_o2 = detect_entity(img_path, o2_name)
            print(f'       → score={score_o2:.3f}, box={box_o2}')

        marks_ok = (box_o1 is not None) or (box_o2 is not None)
        marked_img = draw_bbox_marks(img, box_o1, box_o2, o1_name, o2_name)
        print(f'  [M1] marks_ok={marks_ok}')
    else:
        print(f'  [M1] SKIP (gating_pass=False: conf={conf:.2f}, halluc={halluc})')

    # ── Module 2: Depth ──
    depth_ok = False
    depth_o1 = 0.0
    depth_o2 = 0.0

    if gating_pass:
        print(f'  [M2] Computing depth map...')
        depth_map = compute_depth_map(img)
        depth_o1 = get_mean_depth_in_bbox(depth_map, box_o1, depth_map.shape)
        depth_o2 = get_mean_depth_in_bbox(depth_map, box_o2, depth_map.shape)
        depth_ok = True
        print(f'  [M2] d(O1)={depth_o1:.4f}, d(O2)={depth_o2:.4f}')
    else:
        print(f'  [M2] SKIP (gating_pass=False)')

    # Lưu kết quả
    debug_dir = os.path.join(OUTPUT_DIR, f'{idx}_{sid}')
    os.makedirs(debug_dir, exist_ok=True)
    img.save(os.path.join(debug_dir, 'image_orig.jpg'), quality=95)
    marked_img.save(os.path.join(debug_dir, 'image1_bbox.jpg'), quality=95)

    meta = {
        'id': sid, 'image': img_name,
        'o1': o1_name, 'o2': o2_name, 'is_viewer': is_viewer,
        'conf': conf, 'gating_pass': gating_pass,
        'marks_ok': marks_ok, 'box_o1': box_o1, 'box_o2': box_o2,
        'score_o1': score_o1, 'score_o2': score_o2,
        'depth_ok': depth_ok, 'depth_o1': round(depth_o1, 4), 'depth_o2': round(depth_o2, 4),
    }
    with open(os.path.join(debug_dir, 'meta.json'), 'w') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    results.append(meta)

print(f'\n=== Tóm tắt ===')
print(f'Total: {len(results)}')
print(f'Gating pass: {sum(1 for r in results if r.get("gating_pass"))}')
print(f'M1 marks_ok: {sum(1 for r in results if r.get("marks_ok"))}')
print(f'M2 depth_ok: {sum(1 for r in results if r.get("depth_ok"))}')

## Cell 13 — Visualize kết quả

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

OUTPUT_DIR = '/kaggle/working/ASTRA/output/debug_v2'
subdirs = sorted(os.listdir(OUTPUT_DIR))

for subdir in subdirs[:5]:  # Hiện 5 sample đầu
    debug_path = os.path.join(OUTPUT_DIR, subdir)
    orig_path = os.path.join(debug_path, 'image_orig.jpg')
    bbox_path = os.path.join(debug_path, 'image1_bbox.jpg')
    meta_path = os.path.join(debug_path, 'meta.json')

    if not os.path.exists(orig_path):
        continue

    with open(meta_path) as f:
        meta = json.load(f)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(
        f'id={meta["id"]} | O1: {meta["o1"]} | O2: {meta["o2"]}\n'
        f'marks_ok={meta["marks_ok"]} | depth_o1={meta["depth_o1"]:.3f} | depth_o2={meta["depth_o2"]:.3f}',
        fontsize=11
    )
    axes[0].imshow(mpimg.imread(orig_path))
    axes[0].set_title('Ảnh gốc')
    axes[0].axis('off')

    if os.path.exists(bbox_path):
        axes[1].imshow(mpimg.imread(bbox_path))
        axes[1].set_title('Set-of-Mark (M1)')
    else:
        axes[1].set_title('Không có bbox')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()